# Isolated Mem adapter — full LIBERO-10 + T1/T3
Select GPU T4 ×2 and enable Internet. Attach the generated code and **completed** data packages as private Kaggle datasets. This notebook trains an unmerged adapter from the pinned base. It does not run benchmarks or claim retention/Mem success. Memory inputs were predicted causally offline using the online writer contract.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/mem-adapter-v1')
WORK.mkdir(exist_ok=True)
def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Archive path escapes destination')
        z.extractall(destination)
code_zips = list(INPUT.rglob('mem_adapter_v1_code.zip'))
if len(code_zips) == 1:
    extract_checked(code_zips[0], WORK / 'code')
else:
    candidates = list(INPUT.rglob('train_mem_adapter.py'))
    if len(candidates) != 1:
        raise ValueError('Attach exactly one new Mem adapter code package')
    shutil.copytree(candidates[0].parent.parent, WORK / 'code', dirs_exist_ok=True)
REPO = WORK / 'code'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'envs/mem_adapter_training.txt')], check=True)
# The following commands run in fresh Python processes, avoiding stale imported libraries.
subprocess.run([sys.executable, '-c', 'import torch; assert torch.cuda.device_count() == 2; print(torch.__version__); print([torch.cuda.get_device_name(i) for i in range(2)])'], check=True)


In [ ]:
data_candidates = []
for p in INPUT.rglob('manifest.json'):
    try:
        m = json.loads(p.read_text())
    except (ValueError, OSError):
        continue
    if m.get('artifact') == 'mixed_base_mem_sft_shards' and m.get('memory_source') == 'predicted_rgb_v3' and len(m.get('retention_replay_tasks', [])) == 10:
        data_candidates.append(p.parent)
if len(data_candidates) == 1:
    DATA = data_candidates[0]
else:
    archives = list(INPUT.rglob('mem_adapter_v1_data.zip'))
    if len(archives) != 1:
        raise ValueError('Attach exactly one completed expanded memory dataset')
    import hashlib
    sidecar = archives[0].with_suffix('.json')
    if not sidecar.is_file():
        raise ValueError('Archive SHA256 sidecar is required')
    digest = hashlib.sha256()
    with archives[0].open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            digest.update(block)
    assert digest.hexdigest() == json.loads(sidecar.read_text())['sha256']
    extract_checked(archives[0], WORK / 'data')
    DATA = WORK / 'data/dataset'
print(DATA)


In [ ]:
ASSETS = Path('/kaggle/temp/mem-adapter-assets')
subprocess.run([sys.executable, str(REPO / 'scripts/memory_sft_assets.py'), 'fetch', str(ASSETS)], cwd=REPO, check=True)


In [ ]:
from datetime import datetime, timezone
OUTPUT = Path('/kaggle/working') / ('mem-adapter-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
subprocess.run([sys.executable, str(REPO / 'scripts/train_mem_adapter.py'), '--data-dir', str(DATA), '--assets-dir', str(ASSETS), '--output-dir', str(OUTPUT), '--gpus', '2'], cwd=REPO, check=True)
print('Unmerged adapter:', OUTPUT / 'pretrained_model')
print('Training export complete; policy success remains unmeasured.')


Download the complete output directory including metrics, run_config, checkpoints and pretrained_model. Use the original base checkpoint for baseline tasks. Deploy the adapter through the F1 runner with its verified writer and pinned tokenizer. The final checkpoint is not automatically the best checkpoint.